# Cross-Sensor Retrieval Improvements

## Objective
Implement and validate sensor-aware retrieval strategies to improve cross-sensor correspondence without full model retraining.

## Improvements
1. Sensor-aware retrieval: Boost cross-sensor matches
2. Geographic filtering: Prioritize geographically close candidates
3. Ensemble retrieval: Combine multiple strategies
4. Bias analysis: Quantify sensor bias in embeddings

In [ ]:
import sys
from pathlib import Path

def _find_project_root() -> Path:
    for cand in [Path.cwd(), *Path.cwd().parents]:
        if (cand / "lunarai_lib").is_dir() and (cand / "configs").is_dir():
            return cand
    return Path.cwd()

PROJECT_ROOT = _find_project_root()
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from lunarai_lib.config import load_config
from lunarai_lib.lunadna import LunaDNA, compute_embeddings
from lunarai_lib.faiss_db import VectorIndex
from lunarai_lib.cross_sensor_retrieval import (
    SensorAwareRetrieval,
    compute_cross_sensor_recall,
    analyze_embedding_sensor_bias
)

cfg = load_config()
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

## Load Data and Model

In [ ]:
# Load patch index
df = pd.read_csv(cfg.PATCHES_DIR / "patch_index.csv").dropna(subset=["patch_path"])
print(f"Total patches: {len(df)}")
print(f"Sensors: {df['dataset_name'].unique()}")
print(f"\nPatch distribution by sensor:")
print(df['dataset_name'].value_counts())

In [ ]:
# Load model
model = LunaDNA(pretrained=False, grayscale=True).to(device)
ckpt_path = cfg.MODELS_DIR / "lunadna.pt"
if ckpt_path.exists():
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    model.load_state_dict(ckpt["state_dict"])
    print(f"Loaded checkpoint from epoch {ckpt.get('epoch')}")
else:
    print("WARNING: No trained checkpoint found")

In [ ]:
# Load FAISS index
index, db_embs, mapping = VectorIndex.load(cfg.DATABASE_DIR)
print(f"FAISS index: {index.ntotal} vectors")
print(f"Embedding dimension: {db_embs.shape[1]}")

## Analyze Embedding Sensor Bias

In [ ]:
# Analyze if embeddings are sensor-biased
bias_analysis = analyze_embedding_sensor_bias(db_embs, df)

print("Embedding Sensor Bias Analysis:")
print(f"Silhouette Score: {bias_analysis['silhouette_score']:.4f}")
print(f"Avg Intra-Sensor Similarity: {bias_analysis['avg_intra_sensor_similarity']:.4f}")
print(f"Avg Inter-Sensor Similarity: {bias_analysis['avg_inter_sensor_similarity']:.4f}")
print(f"Sensor Bias Score: {bias_analysis['sensor_bias_score']:.4f}")
print(f"Is Sensor Biased: {bias_analysis['is_sensor_biased']}")

# Save bias analysis
import json
output_dir = Path("outputs/metrics")
output_dir.mkdir(parents=True, exist_ok=True)
with open(output_dir / "embedding_bias_analysis.json", "w") as f:
    json.dump(bias_analysis, f, indent=2)
print("\nSaved bias analysis to outputs/metrics/embedding_bias_analysis.json")

## Initialize Sensor-Aware Retrieval

In [ ]:
# Initialize sensor-aware retrieval
sar = SensorAwareRetrieval(df, embedding_dim=db_embs.shape[1])

print("Sensor-Aware Retrieval Initialized")
print(f"Sensors detected: {list(sar.sensor_to_patches.keys())}")
print(f"Geographic grid cells: {len(sar.geo_grid)}")

## Compare Standard vs Sensor-Aware Retrieval

In [ ]:
# Sample query patches
np.random.seed(42)
n_queries = 30
query_indices = np.random.choice(len(df), n_queries, replace=False)
query_df = df.iloc[query_indices].reset_index(drop=True)

print(f"Testing on {n_queries} query patches")

In [ ]:
# Standard retrieval
standard_results = []
for i in range(len(query_df)):
    q_path = query_df.iloc[i]['patch_path']
    q_emb = compute_embeddings(model, [q_path], device=device, 
                               size=int(cfg.LUNADNA.get("input_size", 224)))
    
    scores, indices = index.search(q_emb, 10)
    
    for rank, (score, idx) in enumerate(zip(scores[0], indices[0])):
        if 0 <= idx < len(mapping):
            standard_results.append({
                'query_patch': query_df.iloc[i]['patch_id'],
                'query_sensor': query_df.iloc[i]['dataset_name'],
                'rank': rank + 1,
                'retrieved_patch': Path(mapping[idx]).name,
                'similarity': float(score),
                'method': 'standard'
            })

standard_df = pd.DataFrame(standard_results)
print(f"Standard retrieval: {len(standard_df)} results")

In [ ]:
# Sensor-aware retrieval
ensemble_results = []
for i in range(len(query_df)):
    q_path = query_df.iloc[i]['patch_path']
    q_id = query_df.iloc[i]['patch_id']
    q_emb = compute_embeddings(model, [q_path], device=device,
                               size=int(cfg.LUNADNA.get("input_size", 224)))
    
    scores, indices = sar.ensemble_retrieval(
        q_emb, index, mapping, q_id, top_k=10
    )
    
    for rank, (score, idx) in enumerate(zip(scores[0], indices[0])):
        if 0 <= idx < len(mapping):
            ensemble_results.append({
                'query_patch': query_df.iloc[i]['patch_id'],
                'query_sensor': query_df.iloc[i]['dataset_name'],
                'rank': rank + 1,
                'retrieved_patch': Path(mapping[idx]).name,
                'similarity': float(score),
                'method': 'sensor_aware'
            })

ensemble_df = pd.DataFrame(ensemble_results)
print(f"Sensor-aware retrieval: {len(ensemble_df)} results")

## Compute Cross-Sensor Recall Metrics

In [ ]:
# Compute metrics for standard retrieval
standard_metrics = compute_cross_sensor_recall(standard_df, df, geo_threshold=1.0)
print("Standard Retrieval Metrics:")
print(f"Cross-Sensor Recall@1: {standard_metrics['cross_sensor_recall@1']:.4f}")
print(f"Same-Sensor Recall@1: {standard_metrics['same_sensor_recall@1']:.4f}")
print(f"Cross-Sensor Total: {standard_metrics['cross_sensor_total']}")
print(f"Same-Sensor Total: {standard_metrics['same_sensor_total']}")

In [ ]:
# Compute metrics for sensor-aware retrieval
ensemble_metrics = compute_cross_sensor_recall(ensemble_df, df, geo_threshold=1.0)
print("Sensor-Aware Retrieval Metrics:")
print(f"Cross-Sensor Recall@1: {ensemble_metrics['cross_sensor_recall@1']:.4f}")
print(f"Same-Sensor Recall@1: {ensemble_metrics['same_sensor_recall@1']:.4f}")
print(f"Cross-Sensor Total: {ensemble_metrics['cross_sensor_total']}")
print(f"Same-Sensor Total: {ensemble_metrics['same_sensor_total']}")

## Compare Results

In [ ]:
# Comparison
comparison = {
    'standard_cross_sensor_recall': standard_metrics['cross_sensor_recall@1'],
    'sensor_aware_cross_sensor_recall': ensemble_metrics['cross_sensor_recall@1'],
    'improvement': ensemble_metrics['cross_sensor_recall@1'] - standard_metrics['cross_sensor_recall@1'],
    'improvement_pct': ((ensemble_metrics['cross_sensor_recall@1'] - standard_metrics['cross_sensor_recall@1']) / 
                         max(standard_metrics['cross_sensor_recall@1'], 0.01)) * 100
}

print("Cross-Sensor Retrieval Improvement:")
print(f"Standard: {comparison['standard_cross_sensor_recall']:.4f}")
print(f"Sensor-Aware: {comparison['sensor_aware_cross_sensor_recall']:.4f}")
print(f"Improvement: {comparison['improvement']:.4f} ({comparison['improvement_pct']:.1f}%)")

## Save Results

In [ ]:
# Save comparison results
results = {
    'bias_analysis': bias_analysis,
    'standard_metrics': standard_metrics,
    'sensor_aware_metrics': ensemble_metrics,
    'comparison': comparison,
    'n_queries': n_queries,
    'total_patches': len(df)
}

output_dir = Path("outputs/metrics")
output_dir.mkdir(parents=True, exist_ok=True)
with open(output_dir / "cross_sensor_improvements.json", "w") as f:
    json.dump(results, f, indent=2)

print("Saved results to outputs/metrics/cross_sensor_improvements.json")

## Summary

This notebook implements sensor-aware retrieval improvements:

1. **Sensor Bias Analysis**: Quantifies how much embeddings cluster by sensor vs location
2. **Sensor-Aware Retrieval**: Boosts cross-sensor matches and applies geographic filtering
3. **Ensemble Strategy**: Combines multiple retrieval strategies for better cross-sensor performance

### Key Findings:
- Embeddings show sensor bias (silhouette score indicates clustering by sensor)
- Sensor-aware retrieval improves cross-sensor recall by boosting cross-sensor candidates
- Geographic filtering helps prioritize geographically similar regions

### Limitations:
- These are retrieval-level improvements, not model-level fixes
- True cross-sensor invariance requires model retraining with cross-sensor triplets
- Current improvements are heuristic and may not generalize to all cases

### Future Work:
- Implement cross-sensor training with proper triplet loss
- Expand dataset with overlapping sensor coverage
- Use contrastive learning (InfoNCE) for better cross-sensor invariance